# Advanced Python Problems with Fully Worked Solutions
## Property lookup resolution · data and non-data descriptors

**Source basis:** The uploaded lesson titled *Property Lookup Resolution*. Its examples involving `IntegerValue`, `TimeUTC`, and `ValidString` motivate these exercises. The problems and solutions below are original extensions, not verbatim copies of the source.

**Audience:** Advanced Python learners; **runtime:** Python 3.10+; **dependencies:** Python standard library only. **Run:** Kernel → Restart & Run All. All cells are ordered, self-checking, and avoid uncontrolled recursion or time-dependent results.

**Study workflow:** Read each challenge, write down a predicted result, inspect the worked solution, execute its code and checks, then try modifying the example. Each problem defines its own classes to avoid accidental dependence on the earlier exercises.

## Compact reference: ordinary instance attribute reads

For `obj.name` under the usual `object.__getattribute__` implementation, the useful mental model is:

1. Find the first `name` in `type(obj).__mro__` (the nearest class attribute).
2. If it is a **data descriptor** (has `__get__` plus `__set__` or `__delete__`), invoke it before consulting the instance dictionary.
3. Otherwise, if `name` exists in `obj.__dict__`, return that value.
4. Otherwise, invoke a class **non-data descriptor** (`__get__` only), or return a plain class attribute.
5. On an `AttributeError`, ordinary `getattr` / dotted access can call an available `__getattr__` fallback.

**Important limits:** Custom `__getattribute__`, `__slots__`, metaclass lookup, and `super()` require extra care; later problems address these separately. Descriptors are identified by methods on their *type*, not by arbitrary names stored in an individual descriptor object's dictionary. A read-only `property` remains a data descriptor even if its setter is missing. Writing straight to `__dict__` bypasses descriptor validation; same-name storage is a convenience, not a security boundary.

In [1]:
from contextlib import contextmanager

@contextmanager
def raises(expected_exception):
    """Minimal standard-library-only assertion helper for negative tests."""
    try:
        yield
    except expected_exception:
        pass
    else:
        raise AssertionError(f"Expected {expected_exception.__name__}")

print("Common test helper ready")

Common test helper ready


## Problem 01 — Trace the complete precedence rule  
**Level:** Advanced

### Challenge
Implement one data descriptor, one non-data descriptor, and one plain class attribute. Give their owning class an instance dictionary containing all three corresponding names. Predict the returned values and which descriptor hooks run. Then remove the instance entries and repeat.

**Before running the solution:** predict the result and identify which lookup step applies.

### Worked solution
The class is searched first to *locate* `data`; because it is a data descriptor, its `__get__` wins over the same-named instance entry. The instance entries for `nondata` and `plain` win instead. `__set__` is invoked by normal assignment even when an instance key already exists. The `events_01` log makes the decision observable without relying on timestamps.

In [2]:
events_01 = []

class DataProbe01:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        events_01.append("data.__get__")
        return "from-data-descriptor"

    def __set__(self, instance, value):
        events_01.append("data.__set__")
        instance.__dict__["data"] = value

class NonDataProbe01:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        events_01.append("nondata.__get__")
        return "from-nondata-descriptor"

class Lookup01:
    data = DataProbe01()
    nondata = NonDataProbe01()
    plain = "class-value"

item_01 = Lookup01()
item_01.__dict__.update(data="raw-data", nondata="raw-nondata", plain="raw-plain")
print(item_01.data, item_01.nondata, item_01.plain)
print("hook events:", events_01)

from-data-descriptor raw-nondata raw-plain
hook events: ['data.__get__']


### Executable checks

In [3]:
assert (item_01.data, item_01.nondata, item_01.plain) == (
    "from-data-descriptor", "raw-nondata", "raw-plain"
)
assert events_01 == ["data.__get__", "data.__get__"]
events_01.clear()
item_01.__dict__.pop("nondata")
item_01.__dict__.pop("plain")
assert item_01.nondata == "from-nondata-descriptor"
assert item_01.plain == "class-value"
assert events_01 == ["nondata.__get__"]
item_01.data = "new"
assert item_01.__dict__["data"] == "new"
assert events_01[-1] == "data.__set__"
print("Problem 01: PASS")

Problem 01: PASS


## Problem 02 — Recognize data descriptors, including read-only properties  
**Level:** Advanced

### Challenge
Write an inspection function that distinguishes a data descriptor, a non-data descriptor, and a non-descriptor. Verify that a `property` without a setter is still a data descriptor. Show what happens when the instance dictionary tries to shadow that property.

**Before running the solution:** predict the result and identify which lookup step applies.

### Worked solution
Classification checks the *descriptor object's type*, including inherited hooks. A deletion hook is sufficient to make a get-capable descriptor a data descriptor. Python's built-in `property` provides write/delete protocol hooks that raise `AttributeError` when no setter/deleter is defined; the missing user-defined setter does **not** turn it into a non-data descriptor.

In [4]:
class GetOnly02:
    def __get__(self, instance, owner):
        return "read" if instance is not None else self

class DeleteOnly02:
    def __get__(self, instance, owner):
        return "protected" if instance is not None else self

    def __delete__(self, instance):
        instance.__dict__.pop("guarded", None)

class Sample02:
    value = GetOnly02()
    guarded = DeleteOnly02()

    @property
    def score(self):
        return 42

    constant = 7

def descriptor_kind(value):
    descriptor_type = type(value)
    has_get = hasattr(descriptor_type, "__get__")
    has_write_hook = (hasattr(descriptor_type, "__set__") or
                      hasattr(descriptor_type, "__delete__"))
    if has_get and has_write_hook:
        return "data"
    if has_get:
        return "non-data"
    return "ordinary"

kinds_02 = {name: descriptor_kind(vars(Sample02)[name])
            for name in ("value", "guarded", "score", "constant")}
print(kinds_02)

{'value': 'non-data', 'guarded': 'data', 'score': 'data', 'constant': 'ordinary'}


### Executable checks

In [5]:
assert kinds_02 == {
    "value": "non-data", "guarded": "data",
    "score": "data", "constant": "ordinary"
}
sample_02 = Sample02()
sample_02.__dict__["score"] = "forged"
assert sample_02.score == 42
with raises(AttributeError):
    sample_02.score = 99
with raises(AttributeError):
    del sample_02.score
print("Problem 02: PASS")

Problem 02: PASS


## Problem 03 — Build a reusable `ValidString` with same-name storage  
**Level:** Advanced

### Challenge
Inspired by the lesson's `ValidString`, implement a descriptor configured by minimum length. Use `__set_name__` to learn the field name. Store the value under that name in each instance's `__dict__`, return the descriptor on class access, and raise `AttributeError` when unset.

**Before running the solution:** predict the result and identify which lookup step applies.

### Worked solution
`__set_name__` runs when the class body is created and supplies each descriptor's own field name. Normal `instance.first_name = ...` calls `__set__`; direct `instance.__dict__[name]` writes inside the descriptor avoid recursive dispatch. A stored key with the same name does **not** shadow the data descriptor on later reads. An unset attribute raises the conventional `AttributeError`, which also allows a containing class's `__getattr__` to act if it defines one.

In [6]:
class ValidString03:
    def __init__(self, min_length):
        if not isinstance(min_length, int) or isinstance(min_length, bool) or min_length < 0:
            raise ValueError("min_length must be a nonnegative integer")
        self.min_length = min_length

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.name]
        except KeyError:
            raise AttributeError(f"{self.name} has not been set") from None

    def __set__(self, instance, value):
        if not isinstance(value, str):
            raise TypeError(f"{self.name} must be str")
        if len(value) < self.min_length:
            raise ValueError(f"{self.name} needs {self.min_length}+ characters")
        instance.__dict__[self.name] = value

class Person03:
    first_name = ValidString03(1)
    last_name = ValidString03(2)

person_03 = Person03()
person_03.first_name = "Ada"
person_03.last_name = "Lovelace"
print(person_03.__dict__, person_03.first_name)

{'first_name': 'Ada', 'last_name': 'Lovelace'} Ada


### Executable checks

In [7]:
assert Person03.first_name is vars(Person03)["first_name"]
assert person_03.__dict__ == {"first_name": "Ada", "last_name": "Lovelace"}
assert (person_03.first_name, person_03.last_name) == ("Ada", "Lovelace")
assert Person03.first_name.name == "first_name"
assert Person03.last_name.name == "last_name"
with raises(AttributeError):
    _ = Person03().first_name
with raises(TypeError):
    person_03.first_name = 100
with raises(ValueError):
    person_03.last_name = "X"
print("Problem 03: PASS")

Problem 03: PASS


## Problem 04 — Prove instance isolation and failure atomicity  
**Level:** Advanced

### Challenge
Extend the preceding problem with test scenarios: two `Person03` objects must not share stored values, and invalid assignment must leave the prior valid value untouched. Distinguish descriptor configuration shared on the class from data stored on each instance.

**Before running the solution:** predict the result and identify which lookup step applies.

### Worked solution
The single descriptor object in `Person03.__dict__` is shared as *configuration*, but it writes into the specific instance passed to `__set__`. Validate **before** changing storage so bad inputs do not partially mutate the object. This illustrates single-field failure atomicity, not a general multi-field transaction guarantee.

In [8]:
left_04 = Person03()
right_04 = Person03()
left_04.first_name = "Grace"
right_04.first_name = "Linus"

before_04 = left_04.__dict__.copy()
try:
    left_04.first_name = ""  # validation precedes mutation
except ValueError as exc:
    print(type(exc).__name__, str(exc))

print(left_04.__dict__, right_04.__dict__)

ValueError first_name needs 1+ characters
{'first_name': 'Grace'} {'first_name': 'Linus'}


### Executable checks

In [9]:
assert left_04.first_name == "Grace"
assert right_04.first_name == "Linus"
assert left_04.__dict__ == before_04
assert left_04.__dict__ is not right_04.__dict__
assert Person03.first_name is vars(Person03)["first_name"]
with raises(TypeError):
    left_04.first_name = None
assert left_04.first_name == "Grace"
print("Problem 04: PASS")

Problem 04: PASS


## Problem 05 — Deletion hooks and a deletion-only data descriptor  
**Level:** Advanced

### Challenge
Implement `__delete__` for a managed attribute and check that it removes the same-named instance key. Build a separate descriptor with `__get__` and `__delete__`, but no `__set__`. Explain why a raw dictionary entry cannot shadow either descriptor on a read.

**Before running the solution:** predict the result and identify which lookup step applies.

### Worked solution
A descriptor with `__get__` and `__delete__` is a data descriptor even without `__set__`. Ordinary assignment to that attribute is rejected, but directly mutating `__dict__` can still insert a key; descriptor precedence makes that key invisible through normal attribute lookup. Deletion of absent managed data raises `AttributeError` consistently.

In [10]:
class Deletable05:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        if self.name not in instance.__dict__:
            raise AttributeError(self.name)
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        instance.__dict__[self.name] = value

    def __delete__(self, instance):
        try:
            del instance.__dict__[self.name]
        except KeyError:
            raise AttributeError(self.name) from None

class DeletionOnly05:
    def __get__(self, instance, owner=None):
        return self if instance is None else "descriptor-wins"

    def __delete__(self, instance):
        instance.__dict__.pop("locked", None)

class Record05:
    field = Deletable05()
    locked = DeletionOnly05()

record_05 = Record05()
record_05.field = "active"
record_05.__dict__["locked"] = "shadow-attempt"
print(record_05.field, record_05.locked)

active descriptor-wins


### Executable checks

In [11]:
assert record_05.locked == "descriptor-wins"
assert record_05.__dict__["locked"] == "shadow-attempt"
with raises(AttributeError):
    record_05.locked = "normal-write"
del record_05.locked
assert "locked" not in record_05.__dict__
del record_05.field
assert "field" not in record_05.__dict__
with raises(AttributeError):
    _ = record_05.field
with raises(AttributeError):
    del record_05.field
print("Problem 05: PASS")

Problem 05: PASS


## Problem 06 — Design for `__slots__` without an instance dictionary  
**Level:** Advanced

### Challenge
The source's same-name storage assumes a writable instance `__dict__`. Implement a strict positive-integer descriptor that stores state in a *separate private slot* and works for objects with no dictionary. Reject booleans as integer inputs.

**Before running the solution:** predict the result and identify which lookup step applies.

### Worked solution
A slot declaration installs a separate class-level slot descriptor for `_age`. The custom data descriptor named `age` writes to this *different* slot, so there is no self-recursion. `object.__setattr__` still respects descriptors; it is safe here only because it targets the distinct `_age` slot. Never assume `instance.__dict__` exists on slotted instances.

In [12]:
class PositiveInt06:
    def __set_name__(self, owner, name):
        self.public_name = name
        self.slot_name = "_" + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return object.__getattribute__(instance, self.slot_name)
        except AttributeError:
            raise AttributeError(f"{self.public_name} is unset") from None

    def __set__(self, instance, value):
        if type(value) is not int or value <= 0:
            raise ValueError(f"{self.public_name} must be a positive int")
        object.__setattr__(instance, self.slot_name, value)

class SlottedPerson06:
    __slots__ = ("_age",)
    age = PositiveInt06()

slotted_06 = SlottedPerson06()
slotted_06.age = 37
print("age:", slotted_06.age, "has __dict__:", hasattr(slotted_06, "__dict__"))

age: 37 has __dict__: False


### Executable checks

In [13]:
assert slotted_06.age == 37
assert not hasattr(slotted_06, "__dict__")
assert SlottedPerson06.age.public_name == "age"
assert SlottedPerson06.age.slot_name == "_age"
with raises(ValueError):
    slotted_06.age = True
with raises(ValueError):
    slotted_06.age = -1
assert slotted_06.age == 37
with raises(AttributeError):
    _ = SlottedPerson06().age
print("Problem 06: PASS")

Problem 06: PASS


## Problem 07 — Allow same-name storage with a slotted class that opts in  
**Level:** Advanced

### Challenge
Construct a class using `__slots__ = ('__dict__',)` so that the `ValidString03` descriptor can use its ordinary same-name storage strategy. Contrast this with the dictionary-free class in Problem 6.

**Before running the solution:** predict the result and identify which lookup step applies.

### Worked solution
`__slots__` and same-name dictionary storage are compatible **only if** the instance actually has a `__dict__` (either declared or inherited). This experiment also exposes an important limitation: direct dictionary writes bypass `__set__`'s validation, even though subsequent reads still invoke `__get__`. Use the descriptor's public API rather than raw dictionaries for normal application updates.

In [14]:
class SlottedWithDict07:
    __slots__ = ("__dict__",)
    name = ValidString03(2)

with_dict_07 = SlottedWithDict07()
with_dict_07.name = "Python"
print(with_dict_07.__dict__)

{'name': 'Python'}


### Executable checks

In [15]:
assert with_dict_07.__dict__ == {"name": "Python"}
assert with_dict_07.name == "Python"
with_dict_07.__dict__["name"] = "Z"  # bypasses validation intentionally
assert with_dict_07.name == "Z"  # still resolves through data descriptor
with raises(ValueError):
    with_dict_07.name = "X"
assert with_dict_07.name == "Z"
print("Problem 07: PASS")

Problem 07: PASS


## Problem 08 — Expose and repair descriptor re-entrancy safely  
**Level:** Advanced

### Challenge
Demonstrate the common bug `setattr(instance, self.name, value)` inside `__set__`, but avoid an uncontrolled recursion traceback. Add a guard that makes the second entry raise an explanatory error. Then implement the correct dictionary-backed setter.

**Before running the solution:** predict the result and identify which lookup step applies.

### Worked solution
`setattr(instance, name, value)` calls the **same data descriptor** again when the name matches. The explicit guard turns the infinite recursion from the source lesson into a controlled, testable example. The repair writes directly to the instance's dictionary. For a slot-backed implementation, write to a *different* backing slot instead.

In [16]:
class BrokenSetter08:
    def __set_name__(self, owner, name):
        self.name = name
        self.active = False

    def __set__(self, instance, value):
        if self.active:
            raise RuntimeError("recursive __set__ detected")
        self.active = True
        try:
            setattr(instance, self.name, value)  # intentionally wrong
        finally:
            self.active = False

    def __get__(self, instance, owner=None):
        return self if instance is None else instance.__dict__.get(self.name)

class Broken08:
    value = BrokenSetter08()

broken_08 = Broken08()
with raises(RuntimeError):
    broken_08.value = 123

class CorrectSetter08:
    def __set_name__(self, owner, name):
        self.name = name

    def __set__(self, instance, value):
        instance.__dict__[self.name] = value

    def __get__(self, instance, owner=None):
        return self if instance is None else instance.__dict__.get(self.name)

class Fixed08:
    value = CorrectSetter08()

fixed_08 = Fixed08()
fixed_08.value = 123
print("fixed value:", fixed_08.value)

fixed value: 123


### Executable checks

In [17]:
assert broken_08.__dict__ == {}
assert not Broken08.value.active
assert fixed_08.value == 123
assert fixed_08.__dict__ == {"value": 123}
fixed_08.value = 456
assert fixed_08.value == 456
print("Problem 08: PASS")

Problem 08: PASS


## Problem 09 — Understand `instance`, `owner`, and inherited class access  
**Level:** Advanced

### Challenge
Instrument `__get__` to observe what Python passes on class access versus instance access. Include a subclass inheriting the descriptor. Confirm that `Child.field` returns the descriptor object while `child.field` receives `owner=Child`.

**Before running the solution:** predict the result and identify which lookup step applies.

### Worked solution
`__set_name__` records the owner that originally **defined** the descriptor. When a subclass inherits it, `__get__` receives the class used for the lookup as `owner`. Handle `instance is None` explicitly so class-level inspection yields the descriptor rather than trying to read nonexistent per-instance state.

In [18]:
class OwnerAware09:
    def __set_name__(self, owner, name):
        self.declared_on = owner
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return (self.declared_on.__name__, owner.__name__, type(instance).__name__)

class Base09:
    field = OwnerAware09()

class Child09(Base09):
    pass

child_09 = Child09()
print("class:", Child09.field)
print("instance:", child_09.field)

class: <__main__.OwnerAware09 object at 0x000001B0C199C1A0>
instance: ('Base09', 'Child09', 'Child09')


### Executable checks

In [19]:
assert Base09.field is Child09.field
assert Base09.field.declared_on is Base09
assert Base09.field.name == "field"
assert child_09.field == ("Base09", "Child09", "Child09")
assert Base09().field == ("Base09", "Base09", "Base09")
assert vars(Base09)["field"] is Base09.field
print("Problem 09: PASS")

Problem 09: PASS


## Problem 10 — Handle descriptors attached after class creation  
**Level:** Advanced

### Challenge
Show that class creation automatically calls `__set_name__`, but assigning a descriptor to an existing class later does not. Repair the dynamically attached descriptor by calling `__set_name__` explicitly. Also demonstrate why reusing one name-tracking descriptor under two field names is unsafe.

**Before running the solution:** predict the result and identify which lookup step applies.

### Worked solution
The `type` machinery invokes `__set_name__` while constructing a class, not on arbitrary later assignments. The same descriptor instance assigned twice receives multiple naming callbacks, leaving one shared `name` field pointing to the last name. Create independent descriptor instances per field, or explicitly design per-owner/per-name bookkeeping when sharing is required.

In [20]:
class Named10:
    def __init__(self):
        self.name = None

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        if self.name is None:
            raise RuntimeError("__set_name__ not initialized")
        return instance.__dict__.get(self.name, "unset")

class Dynamic10:
    pass

late_10 = Named10()
Dynamic10.token = late_10
with raises(RuntimeError):
    _ = Dynamic10().token
late_10.__set_name__(Dynamic10, "token")
dynamic_10 = Dynamic10()
dynamic_10.__dict__["token"] = "ready"
print(dynamic_10.token)

shared_10 = Named10()
class Duplicate10:
    left = shared_10
    right = shared_10

class Independent10:
    left = Named10()
    right = Named10()

ready


### Executable checks

In [21]:
assert dynamic_10.token == "ready"
assert Dynamic10.token is late_10
assert Duplicate10.left is Duplicate10.right
assert shared_10.name == "right"
assert Independent10.left is not Independent10.right
assert (Independent10.left.name, Independent10.right.name) == ("left", "right")
print("Problem 10: PASS")

Problem 10: PASS


## Problem 11 — Implement a cache using a non-data descriptor  
**Level:** Advanced

### Challenge
Create a lazily computed, per-instance cached value using **only** `__get__` and `__set_name__`. Have the descriptor write its result to the instance dictionary under the public field name; show that future reads bypass `__get__`, that deleting the cached key recomputes it, and that instances have separate caches.

**Before running the solution:** predict the result and identify which lookup step applies.

### Worked solution
Because there is no `__set__` or `__delete__`, `CachedValue11` is non-data. Its first `__get__` writes a same-name instance attribute that *shadows the descriptor* on every later read. This is the opposite of the `ValidString03` data-descriptor pattern. Invalidation is crucial when underlying inputs can change; the capstone implements it explicitly.

In [22]:
class CachedValue11:
    def __init__(self):
        self.computations = 0

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        self.computations += 1
        result = f"generated-{self.computations}"
        instance.__dict__[self.name] = result
        return result

class Service11:
    token = CachedValue11()

first_11, second_11 = Service11(), Service11()
print(first_11.token, first_11.token, second_11.token)

generated-1 generated-1 generated-2


### Executable checks

In [23]:
assert first_11.token == "generated-1"
assert second_11.token == "generated-2"
assert Service11.token.computations == 2
assert first_11.__dict__["token"] == "generated-1"
del first_11.token  # deletes the shadowing instance entry
assert first_11.token == "generated-3"
assert Service11.token.computations == 3
assert second_11.token == "generated-2"
first_11.token = "manual-override"
assert first_11.token == "manual-override"
assert Service11.token.computations == 3
print("Problem 11: PASS")

Problem 11: PASS


## Problem 12 — Explain why a read-only `property` wins over an instance key  
**Level:** Advanced

### Challenge
Use the built-in `property` to expose computed `total` as read-only. Inject a key named `total` directly into the instance dictionary and test normal read, assignment, deletion, and the raw stored value.

**Before running the solution:** predict the result and identify which lookup step applies.

### Worked solution
Even a read-only property is a data descriptor, so normal lookup invokes the getter rather than returning the forged instance value. Rejected assignments and deletions are property protocol behavior, not evidence that no `__set__`/`__delete__` mechanism exists. Raw dictionary editing remains possible unless the object design prevents it; attribute lookup precedence alone is not encapsulation against hostile callers.

In [24]:
class Invoice12:
    def __init__(self, cents):
        self._cents = cents

    @property
    def total(self):
        return self._cents / 100

invoice_12 = Invoice12(1250)
invoice_12.__dict__["total"] = "not-the-total"
print("attribute:", invoice_12.total)
print("raw dictionary:", invoice_12.__dict__["total"])

attribute: 12.5
raw dictionary: not-the-total


### Executable checks

In [25]:
assert invoice_12.total == 12.5
assert invoice_12.__dict__["total"] == "not-the-total"
assert descriptor_kind(vars(Invoice12)["total"]) == "data"
with raises(AttributeError):
    invoice_12.total = 100
with raises(AttributeError):
    del invoice_12.total
assert invoice_12.total == 12.5
print("Problem 12: PASS")

Problem 12: PASS


## Problem 13 — Show that ordinary instance methods are shadowable  
**Level:** Advanced

### Challenge
Prove that a function stored in a class is a non-data descriptor that produces a bound method when found through an instance. Shadow the method with a callable stored in the instance dictionary. Access the unshadowed class function explicitly and restore normal binding.

**Before running the solution:** predict the result and identify which lookup step applies.

### Worked solution
A Python function defined in a class implements `__get__` to bind `self`, but no descriptor write hook. An instance key therefore takes precedence and may hold a completely different callable. The callable retrieved from the instance dictionary is returned as-is; it does not receive automatic method binding.

In [26]:
class Worker13:
    def calculate(self, n):
        return n * 2

worker_13 = Worker13()
original_13 = worker_13.calculate
print("bound method result:", original_13(5))
worker_13.__dict__["calculate"] = lambda n: n + 100
print("instance override:", worker_13.calculate(5))

bound method result: 10
instance override: 105


### Executable checks

In [27]:
assert original_13.__self__ is worker_13
assert original_13(5) == 10
assert worker_13.calculate(5) == 105
assert Worker13.calculate(worker_13, 5) == 10
assert descriptor_kind(vars(Worker13)["calculate"]) == "non-data"
del worker_13.calculate
assert worker_13.calculate(5) == 10
assert "calculate" not in worker_13.__dict__
print("Problem 13: PASS")

Problem 13: PASS


## Problem 14 — Account for MRO before descriptor precedence  
**Level:** Advanced

### Challenge
A base class has a data descriptor named `field`. A subclass defines a plain attribute with the same name; another subclass defines a non-data descriptor with that name. Predict which *class-level candidate* is found before applying descriptor-versus-instance precedence.

**Before running the solution:** predict the result and identify which lookup step applies.

### Worked solution
**MRO lookup picks the nearest class attribute first.** Only that found object is classified as a data descriptor, non-data descriptor, or plain attribute. A more distant base-class data descriptor does not override a same-name attribute defined directly on the subclass. Once the subclass's plain/non-data candidate is selected, a matching instance dictionary key can shadow it.

In [28]:
class BaseData14:
    def __get__(self, instance, owner=None):
        return self if instance is None else "base-data"

    def __set__(self, instance, value):
        instance.__dict__["field"] = value

class Base14:
    field = BaseData14()

class PlainChild14(Base14):
    field = "subclass-plain"

class NonData14:
    def __get__(self, instance, owner=None):
        return self if instance is None else "subclass-nondata"

class DescriptorChild14(Base14):
    field = NonData14()

base_14 = Base14()
plain_14 = PlainChild14()
nondata_14 = DescriptorChild14()
for item in (base_14, plain_14, nondata_14):
    item.__dict__["field"] = "instance-field"
print(base_14.field, plain_14.field, nondata_14.field)

base-data instance-field instance-field


### Executable checks

In [29]:
assert base_14.field == "base-data"
assert plain_14.field == "instance-field"
assert nondata_14.field == "instance-field"
del plain_14.__dict__["field"]
del nondata_14.__dict__["field"]
assert plain_14.field == "subclass-plain"
assert nondata_14.field == "subclass-nondata"
assert Base14.field is vars(Base14)["field"]
print("Problem 14: PASS")

Problem 14: PASS


## Problem 15 — Separate `__getattribute__` from `__getattr__` fallback  
**Level:** Advanced

### Challenge
Demonstrate that overriding `__getattribute__` can intentionally (or accidentally) bypass data descriptor precedence. Implement a correct delegation using `object.__getattribute__` and show that `__getattr__` handles both a missing name and a descriptor getter that raises `AttributeError`.

**Before running the solution:** predict the result and identify which lookup step applies.

### Worked solution
The precedence model describes the **default** attribute getter, not every custom implementation. An override that looks into `__dict__` first can break a data descriptor's behavior. The usual safe override delegates to `object.__getattribute__`. `__getattr__` is a fallback on failed *ordinary* access; directly calling `object.__getattribute__` is not the same as a full dotted lookup with fallback.

In [30]:
class FixedData15:
    def __get__(self, instance, owner=None):
        return self if instance is None else "descriptor-answer"

    def __set__(self, instance, value):
        instance.__dict__["field"] = value

class Unsafe15:
    field = FixedData15()

    def __getattribute__(self, name):
        d = object.__getattribute__(self, "__dict__")
        if name in d:
            return d[name]  # deliberately bypasses descriptor precedence
        return object.__getattribute__(self, name)

class MissingGetter15:
    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        raise AttributeError("descriptor has no value")

class Delegating15:
    field = FixedData15()
    unavailable = MissingGetter15()

    def __getattribute__(self, name):
        return object.__getattribute__(self, name)

    def __getattr__(self, name):
        return f"fallback:{name}"

unsafe_15, safe_15 = Unsafe15(), Delegating15()
unsafe_15.__dict__["field"] = "raw"
safe_15.__dict__["field"] = "raw"
print(unsafe_15.field, safe_15.field, safe_15.unavailable)

raw descriptor-answer fallback:unavailable


### Executable checks

In [31]:
assert unsafe_15.field == "raw"
assert safe_15.field == "descriptor-answer"
assert safe_15.unavailable == "fallback:unavailable"
assert safe_15.missing == "fallback:missing"
assert object.__getattribute__(safe_15, "field") == "descriptor-answer"
with raises(AttributeError):
    object.__getattribute__(safe_15, "missing")
print("Problem 15: PASS")

Problem 15: PASS


## Problem 16 — Implement a restricted lookup oracle  
**Level:** Advanced

### Challenge
Write a small pure-Python model of normal instance reads. Find the nearest class attribute by MRO, resolve data descriptors before `__dict__`, then consider instance entries, non-data descriptors, plain class values, and missing names. Compare model results with built-in `getattr` on a constrained class.

**Before running the solution:** predict the result and identify which lookup step applies.

### Worked solution
The model deliberately covers **get-capable** data/non-data descriptors and ordinary dictionary-backed instances. It omits `__slots__`, custom `__getattribute__`/`__getattr__`, special/metaclass machinery, and descriptors with no `__get__`. Its value is conceptual: predicting real behavior within explicit assumptions. A model claiming to reproduce all of CPython's lookup logic in a few lines would be misleading.

In [32]:
_missing_16 = object()

def lookup_model_16(instance, name):
    """Models default reads only: ordinary __dict__, no custom hooks/slots/metaclass."""
    owner = type(instance)
    class_value = _missing_16
    for cls in owner.__mro__:
        if name in vars(cls):
            class_value = vars(cls)[name]
            break

    if class_value is not _missing_16:
        kind = descriptor_kind(class_value)
        if kind == "data":
            return class_value.__get__(instance, owner)

    instance_vars = vars(instance)
    if name in instance_vars:
        return instance_vars[name]

    if class_value is not _missing_16:
        if descriptor_kind(class_value) == "non-data":
            return class_value.__get__(instance, owner)
        return class_value
    raise AttributeError(name)

class Data16:
    def __get__(self, instance, owner=None):
        return self if instance is None else "data-result"

    def __set__(self, instance, value):
        instance.__dict__["data"] = value

class NonData16:
    def __get__(self, instance, owner=None):
        return self if instance is None else "nondata-result"

class Model16:
    data = Data16()
    nondata = NonData16()
    plain = "class-plain"

model_16 = Model16()
model_16.__dict__.update(data="shadow-data", plain="shadow-plain", local="only-instance")
print({key: lookup_model_16(model_16, key)
       for key in ("data", "nondata", "plain", "local")})

{'data': 'data-result', 'nondata': 'nondata-result', 'plain': 'shadow-plain', 'local': 'only-instance'}


### Executable checks

In [33]:
for name in ("data", "nondata", "plain", "local"):
    assert lookup_model_16(model_16, name) == getattr(model_16, name)
model_16.__dict__["nondata"] = "now-shadowed"
assert lookup_model_16(model_16, "nondata") == model_16.nondata
with raises(AttributeError):
    lookup_model_16(model_16, "absent")
with raises(AttributeError):
    getattr(model_16, "absent")
print("Problem 16: PASS")

Problem 16: PASS


## Problem 17 — Store slotted-object state with weak references  
**Level:** Advanced

### Challenge
Make an alternative to private slot storage using `weakref.WeakKeyDictionary`: descriptor values are keyed by instances rather than by names in `__dict__`. Use a slotted, weak-referenceable, hashable class and show that stored state disappears after its owner is garbage-collected.

**Before running the solution:** predict the result and identify which lookup step applies.

### Worked solution
External storage works without `__dict__`, but `WeakKeyDictionary` requires weak-referenceable and hashable instances; slot-only classes typically need `__weakref__` explicitly. A weak key alone does not prevent leaks if the **stored value** strongly references its key. For a fixed set of fields, an explicit private slot (Problem 6) is often simpler.

In [34]:
import gc
import weakref

class ExternalStorage17:
    def __init__(self):
        self.values = weakref.WeakKeyDictionary()

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return self.values[instance]
        except KeyError:
            raise AttributeError("value is unset") from None

    def __set__(self, instance, value):
        if type(value) is not int:
            raise TypeError("value must be exactly int")
        self.values[instance] = value

class Slotted17:
    __slots__ = ("__weakref__",)
    value = ExternalStorage17()

owner_17 = Slotted17()
owner_17.value = 81
weak_17 = weakref.ref(owner_17)
print(owner_17.value, "dictionary available?", hasattr(owner_17, "__dict__"))

81 dictionary available? False


### Executable checks

In [35]:
assert owner_17.value == 81
assert len(Slotted17.value.values) == 1
assert not hasattr(owner_17, "__dict__")
with raises(TypeError):
    owner_17.value = True
assert owner_17.value == 81
del owner_17
gc.collect()
assert weak_17() is None
assert len(Slotted17.value.values) == 0
with raises(AttributeError):
    _ = Slotted17().value
print("Problem 17: PASS")

Problem 17: PASS


## Problem 18 — Inspect descriptors without triggering their getters  
**Level:** Advanced

### Challenge
Build a getter with an observable side effect. Compare `getattr(obj, 'field')` with `inspect.getattr_static(obj, 'field')`, including a same-named instance dictionary entry. Repeat with a non-data descriptor to see which value static lookup reports.

**Before running the solution:** predict the result and identify which lookup step applies.

### Worked solution
`inspect.getattr_static` is useful for debugging and tooling because it avoids ordinary dynamic descriptor execution. It is **not** a replacement for `getattr` when an application actually needs the computed attribute, and static lookup can return raw descriptor objects that require explicit interpretation.

In [36]:
import inspect

calls_18 = []

class WatchedData18:
    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        calls_18.append("data getter")
        return "computed"

    def __set__(self, instance, value):
        instance.__dict__["field"] = value

class WatchedNonData18:
    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        calls_18.append("non-data getter")
        return "lazy"

class Inspection18:
    field = WatchedData18()
    optional = WatchedNonData18()

inspection_18 = Inspection18()
inspection_18.__dict__.update(field="raw-field", optional="raw-optional")
raw_18 = inspect.getattr_static(inspection_18, "field")
print("static data:", type(raw_18).__name__)
print("normal data:", getattr(inspection_18, "field"))

static data: WatchedData18
normal data: computed


### Executable checks

In [37]:
assert raw_18 is vars(Inspection18)["field"]
assert calls_18 == ["data getter"]
assert inspect.getattr_static(inspection_18, "optional") == "raw-optional"
assert inspection_18.optional == "raw-optional"
assert calls_18 == ["data getter"]
del inspection_18.__dict__["optional"]
assert inspect.getattr_static(inspection_18, "optional") is vars(Inspection18)["optional"]
assert inspection_18.optional == "lazy"
assert calls_18 == ["data getter", "non-data getter"]
print("Problem 18: PASS")

Problem 18: PASS


## Problem 19 — Build an integrated validated profile with cache invalidation  
**Level:** Capstone

### Challenge
Create two validating **data** descriptors (`given_name`, `family_name`) plus one cached **non-data** descriptor (`label`). Every successful name mutation must invalidate the cached label, while invalid mutation must preserve both old data and cache. Demonstrate why direct dictionary writes can make caches stale.

**Before running the solution:** predict the result and identify which lookup step applies.

### Worked solution
This is the key architectural connection: data descriptors retain control of public name assignments, while a non-data descriptor can cache its output by letting an instance key shadow it. Invalidate only **after** successful validation and mutation. The deliberate raw-dictionary bypass shows why this mechanism assumes clients follow your public mutation API; it cannot guarantee coherence against direct storage edits.

In [38]:
class NameField19:
    def __init__(self, min_length):
        self.min_length = min_length

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.name]
        except KeyError:
            raise AttributeError(self.name) from None

    def __set__(self, instance, value):
        if not isinstance(value, str):
            raise TypeError(f"{self.name} must be str")
        if len(value.strip()) < self.min_length:
            raise ValueError(f"{self.name} is too short")
        instance.__dict__[self.name] = value.strip()
        instance.__dict__.pop("label", None)

class CachedLabel19:
    def __init__(self):
        self.computations = 0

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        self.computations += 1
        result = f"{instance.given_name} {instance.family_name}"
        instance.__dict__[self.name] = result
        return result

class Profile19:
    given_name = NameField19(1)
    family_name = NameField19(2)
    label = CachedLabel19()

    def __init__(self, given_name, family_name):
        self.given_name = given_name
        self.family_name = family_name

profile_19 = Profile19("Ada", "Lovelace")
print(profile_19.label, profile_19.label)

Ada Lovelace Ada Lovelace


### Executable checks

In [39]:
assert profile_19.label == "Ada Lovelace"
assert Profile19.label.computations == 1
with raises(ValueError):
    profile_19.family_name = "Q"
assert profile_19.label == "Ada Lovelace"
assert Profile19.label.computations == 1
profile_19.given_name = "Augusta"
assert "label" not in profile_19.__dict__
assert profile_19.label == "Augusta Lovelace"
assert Profile19.label.computations == 2
profile_19.__dict__["given_name"] = "Bypass"
assert profile_19.given_name == "Bypass"
assert profile_19.label == "Augusta Lovelace"  # stale cache: bypassed invalidation
profile_19.given_name = "Ada"  # validated assignment restores coherence
assert profile_19.label == "Ada Lovelace"
assert Profile19.label.computations == 3
print("Problem 19: PASS")

Problem 19: PASS


## Problem 20 — Make cross-field updates transactional  
**Level:** Capstone

### Challenge
A range must satisfy `start <= end`. Enforce this for single-field descriptor assignments while providing a `replace(start, end)` method that validates the *complete proposed state* before committing both fields. Show why a legitimate transition can be impossible if updates must happen in the wrong order.

**Before running the solution:** predict the result and identify which lookup step applies.

### Worked solution
Descriptor-based per-field validation cannot always express a *two-field* state change using intermediate states. `replace` checks the complete candidate, then updates both raw fields only after passing validation; it intentionally bypasses the field setters to avoid intermediate invalidity. This provides failure atomicity for synchronous validation errors, **not** isolation or thread-safety; use locking for concurrent updates if needed.

In [40]:
class Endpoint20:
    def __set_name__(self, owner, name):
        self.name = name
        self.other = "end" if name == "start" else "start"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.name]
        except KeyError:
            raise AttributeError(self.name) from None

    def __set__(self, instance, value):
        if type(value) is not int:
            raise TypeError("endpoints must be integers, excluding bool")
        other = instance.__dict__.get(self.other)
        if other is not None:
            start = value if self.name == "start" else other
            end = value if self.name == "end" else other
            if start > end:
                raise ValueError("start must not exceed end")
        instance.__dict__[self.name] = value

class Window20:
    start = Endpoint20()
    end = Endpoint20()

    def __init__(self, start, end):
        self.replace(start, end)

    def replace(self, start, end):
        if type(start) is not int or type(end) is not int:
            raise TypeError("both endpoints must be integers, excluding bool")
        if start > end:
            raise ValueError("start must not exceed end")
        # Deliberate direct storage after *both* fields have passed validation.
        self.__dict__.update(start=start, end=end)

window_20 = Window20(0, 10)
with raises(ValueError):
    window_20.start = 20
window_20.replace(20, 30)
print(window_20.start, window_20.end)

20 30


### Executable checks

In [41]:
assert (window_20.start, window_20.end) == (20, 30)
before_20 = window_20.__dict__.copy()
with raises(ValueError):
    window_20.replace(50, 40)
with raises(TypeError):
    window_20.replace(True, 100)
assert window_20.__dict__ == before_20
with raises(ValueError):
    window_20.end = 19
assert (window_20.start, window_20.end) == (20, 30)
window_20.end = 40
assert window_20.end == 40
print("Problem 20: PASS")

Problem 20: PASS


## Problem 21 — Use `super()` to bypass instance shadowing  
**Level:** Advanced

### Challenge
Create a base non-data descriptor and a subclass method that accesses it through `super()`. Add a same-named instance attribute. Compare ordinary `obj.message` (shadowed) and `super().message` (base descriptor lookup).

**Before running the solution:** predict the result and identify which lookup step applies.

### Worked solution
`super()` is not synonymous with ordinary instance lookup: it searches a selected portion of the class MRO and handles a descriptor it finds there. The instance dictionary does not win this direct `super().message` access, even though it wins the normal access to the non-data descriptor. This is particularly useful when debugging method or descriptor overrides.

In [42]:
class BaseMessage21:
    def __get__(self, instance, owner=None):
        return self if instance is None else f"base-for-{type(instance).__name__}"

class Base21:
    message = BaseMessage21()

class Child21(Base21):
    def through_super(self):
        return super().message

child_21 = Child21()
child_21.__dict__["message"] = "instance-message"
print("ordinary:", child_21.message)
print("super:", child_21.through_super())

ordinary: instance-message
super: base-for-Child21


### Executable checks

In [43]:
assert child_21.message == "instance-message"
assert child_21.through_super() == "base-for-Child21"
assert Base21.message is vars(Base21)["message"]
del child_21.__dict__["message"]
assert child_21.message == "base-for-Child21"
print("Problem 21: PASS")

Problem 21: PASS


## Problem 22 — Distinguish class lookup from instance lookup with a metaclass  
**Level:** Expert

### Challenge
Define a metaclass with a read-only property named `tag`, and define a class attribute with exactly the same name. Compare `Product.tag`, `Product().tag`, and the unprocessed raw class namespace. Explain why instance attribute rules alone do not predict class attribute reads.

**Before running the solution:** predict the result and identify which lookup step applies.

### Worked solution
Classes themselves are instances of their metaclasses. Consequently, `Product22.tag` follows lookup rules for a **class object**, where a data descriptor on `Meta22` can outrank the class's own namespace. `Product22().tag` follows the *ordinary instance* rules and sees the `Product22` namespace value. Never use the ordinary-object precedence sketch unmodified to explain all class-level lookups.

In [44]:
class Meta22(type):
    @property
    def tag(cls):
        return "metaclass-data-descriptor"

class Product22(metaclass=Meta22):
    tag = "ordinary-class-value"

product_22 = Product22()
print("class read:", Product22.tag)
print("instance read:", product_22.tag)
print("class namespace:", vars(Product22)["tag"])

class read: metaclass-data-descriptor
instance read: ordinary-class-value
class namespace: ordinary-class-value


### Executable checks

In [45]:
assert Product22.tag == "metaclass-data-descriptor"
assert product_22.tag == "ordinary-class-value"
assert vars(Product22)["tag"] == "ordinary-class-value"
with raises(AttributeError):
    Product22.tag = "reassignment"
assert Product22.tag == "metaclass-data-descriptor"
assert Product22().tag == "ordinary-class-value"
print("Problem 22: PASS")

Problem 22: PASS


## Problem 23 — Run a deterministic mutation matrix  
**Level:** Advanced

### Challenge
Stress-test the central contrast across many values, without installing property-based testing libraries. For each iteration, insert matching raw instance keys, verify that the data descriptor still wins and the non-data descriptor is shadowed, then remove the non-data key and confirm its `__get__` becomes active.

**Before running the solution:** predict the result and identify which lookup step applies.

### Worked solution
Seeded tests are repeatable and cover falsy values (`None`, `0`, `False`), tuples, and numeric inputs, rather than relying on just one illustrative string. This is not a proof of Python's implementation, but it tests our examples' observable contract across a wider set of state transitions. For production, keep deterministic regression tests and consider Hypothesis as an *optional* external extension.

In [46]:
import random

class ConstantData23:
    def __get__(self, instance, owner=None):
        return self if instance is None else "DATA"

    def __set__(self, instance, value):
        instance.__dict__["data"] = value

class ConstantNonData23:
    def __get__(self, instance, owner=None):
        return self if instance is None else "NONDATA"

class Host23:
    data = ConstantData23()
    lazy = ConstantNonData23()

rng_23 = random.Random(20260922)
case_count_23 = 80
for _ in range(case_count_23):
    obj = Host23()
    raw_data = rng_23.randint(-10_000, 10_000)
    raw_lazy = rng_23.choice([None, "override", 0, False, (1, 2)])
    obj.__dict__.update(data=raw_data, lazy=raw_lazy)
    assert obj.data == "DATA"
    assert obj.lazy == raw_lazy
    assert obj.__dict__["data"] == raw_data
    del obj.__dict__["lazy"]
    assert obj.lazy == "NONDATA"
print(f"{case_count_23} seeded cases checked")

80 seeded cases checked


### Executable checks

In [47]:
assert case_count_23 == 80
assert descriptor_kind(Host23.data) == "data"
assert descriptor_kind(Host23.lazy) == "non-data"
extra_23 = Host23()
extra_23.data = 999
assert extra_23.__dict__["data"] == 999
assert extra_23.data == "DATA"
print("Problem 23: PASS")

Problem 23: PASS


---
## Synthesis: common traps and correct strategies

| Situation | Result or practice |
|---|---|
| Data descriptor and same-named instance key | Data descriptor wins on ordinary reads. |
| Non-data descriptor and same-named instance key | Instance key wins. |
| Read-only `property` | Still a data descriptor; normal writes/deletes raise. |
| `__get__` + `__delete__`, no `__set__` | Still a data descriptor; normal assignment is rejected. |
| `setattr(instance, field_name, value)` inside that field's `__set__` | Re-enters the descriptor; use direct `__dict__` storage or a *different* private slot. |
| `__slots__` without an instance `__dict__` | Same-name `__dict__` storage is unavailable; use private slots or well-designed external storage. |
| Descriptor inherited from base vs same-named subclass attribute | MRO selects the subclass attribute before descriptor classification. |
| Descriptor dynamically assigned to existing class | Call `__set_name__` manually if its implementation needs it. |
| Direct `__dict__` modification | Bypasses validation and cache invalidation; not a protection boundary. |
| Non-data cached property | Its stored instance entry intentionally shadows future descriptor reads. |

**Final reasoning exercise:** Explain, in your own words, why Problems 3 and 11 both save values under their public descriptor names yet behave oppositely on their next read. Answer: their descriptor types differ in whether a write hook (`__set__` or `__delete__`) exists. Also explain the assumption: normal attribute access on dictionary-backed instances without an overriding `__getattribute__`.

**Completion criterion:** Restart the kernel and run all cells; each problem must print `PASS`, and the final checks below must succeed.

In [48]:
# Notebook-wide, dependency-free completion checks.
assert case_count_23 == 80
assert profile_19.label == "Ada Lovelace"
assert (window_20.start, window_20.end) == (20, 40)
assert len(Slotted17.value.values) == 0
print("ALL 23 PROBLEMS COMPLETED SUCCESSFULLY")

ALL 23 PROBLEMS COMPLETED SUCCESSFULLY
